In [ ]:
pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.4 MB/s eta 0:00:00


In [ ]:
import ee
ee.Authenticate()

In [ ]:
import ee
import pandas as pd

ee.Initialize(project='forest-data-pakistan')

# Pakistan boundary
pakistan = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "Pakistan"))
)

# Hansen dataset
hansen = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")

tree_cover = hansen.select("treecover2000")
loss_year = hansen.select("lossyear")

# Forest mask (>30%)
forest2000 = tree_cover.gt(30)

# Mask loss to forest only
masked_loss = loss_year.updateMask(forest2000)

# Pixel area
loss_area = ee.Image.pixelArea().addBands(masked_loss)

# Group by loss year
stats = loss_area.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName="year"
    ),
    geometry=pakistan.geometry(),
    scale=30,
    maxPixels=1e13
)

groups = stats.get("groups").getInfo()

records = []

for item in groups:
    records.append({
        "year": item["year"] + 2000,
        "loss_sqkm": item["sum"] / 1e6
    })

df = pd.DataFrame(records).sort_values("year")

print(df)

df.to_csv("pakistan_forest_loss_by_year.csv", index=False)

    year  loss_sqkm
0   2001   2.207865
1   2002   3.325810
2   2003   6.492486
3   2004  11.742377
4   2005   9.525829
5   2006  13.453359
6   2007  11.302098
7   2008  10.688205
8   2009   9.312492
9   2010   7.191358
10  2011   4.686890
11  2012   2.405346
12  2013   0.598089
13  2014   0.387321
14  2015   0.019952
15  2017   1.931758
16  2018   0.753423
17  2019   0.815500
18  2020   0.694376
19  2021   0.634753
20  2022   0.481514
21  2023   1.495451
22  2024   1.417648


In [ ]:
tree_cover = hansen.select("treecover2000")
loss_year = hansen.select("lossyear")
gain = hansen.select("gain")

forest_mask = tree_cover.gte(30)

loss_pixels = (
    loss_year
    .updateMask(loss_year.gt(0))
    .updateMask(forest_mask)
)

combined = ee.Image.cat([
    tree_cover.rename("treecover2000"),
    loss_year.rename("lossyear"),
    gain.rename("gain")
]).updateMask(loss_pixels)

points = combined.sample(
    region=pakistan.geometry(),
    scale=30,
    geometries=True
)

def add_coords(feature):
    coords = feature.geometry().coordinates()

    return feature.set({
        "longitude": coords.get(0),
        "latitude": coords.get(1),
        "year": ee.Number(feature.get("lossyear")).add(2000),
        "loss": 1
    })

final_points = points.map(add_coords)

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=final_points,
    description="forest_loss_with_coordinates",
    fileFormat="CSV"
)

task.start()

print("Export started")

Export started


In [ ]:
import pandas as pd
import geopandas as gpd

loss = pd.read_csv(
    "/content/drive/MyDrive/forest_loss_with_coordinates.csv"
)

print(loss.head())

   system:index  gain   latitude  longitude  loss  lossyear  treecover2000  \
0             0     0  36.382706  74.347507     1        21             76   
1             1     0  36.379741  74.345890     1        19             66   
2             2     0  36.373273  74.339961     1        19             74   
3             3     0  36.373004  74.340500     1        19             74   
4             4     0  36.266014  72.236017     1         8             45   

   year                                               .geo  
0  2021  {"geodesic":false,"type":"Point","coordinates"...  
1  2019  {"geodesic":false,"type":"Point","coordinates"...  
2  2019  {"geodesic":false,"type":"Point","coordinates"...  
3  2019  {"geodesic":false,"type":"Point","coordinates"...  
4  2008  {"geodesic":false,"type":"Point","coordinates"...  


In [ ]:
gdf_loss = gpd.GeoDataFrame(
    loss,
    geometry=gpd.points_from_xy(
        loss.longitude,
        loss.latitude
    ),
    crs="EPSG:4326"
)

In [ ]:
pk = gpd.read_file(
    "/content/drive/MyDrive/pak admin 2"
)

print(pk.columns)
print(pk.head())

Index(['adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode',
       'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode',
       'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode',
       'valid_on', 'valid_to', 'area_sqkm', 'cod_versio', 'lang', 'lang1',
       'lang2', 'lang3', 'adm2_ref_n', 'center_lat', 'center_lon', 'geometry'],
      dtype='object')
       adm2_name adm2_name1 adm2_name2 adm2_name3 adm2_pcode     adm1_name  \
0           Bagh       None       None       None      PK101  Azad Kashmir   
1        Bhimber       None       None       None      PK102  Azad Kashmir   
2  Jhelum Valley       None       None       None      PK103  Azad Kashmir   
3         Haveli       None       None       None      PK104  Azad Kashmir   
4          Kotli       None       None       None      PK105  Azad Kashmir   

  adm1_name1 adm1_name2 adm1_name3 adm1_pcode  ...    area_sqkm cod_versio  \
0       None       None       None        PK1  .

In [ ]:
print(gdf_loss.crs)
print(pk.crs)

EPSG:4326
EPSG:4326


In [ ]:
print(pk.head())

print(pk.total_bounds)

       adm2_name adm2_name1 adm2_name2 adm2_name3 adm2_pcode     adm1_name  \
0           Bagh       None       None       None      PK101  Azad Kashmir   
1        Bhimber       None       None       None      PK102  Azad Kashmir   
2  Jhelum Valley       None       None       None      PK103  Azad Kashmir   
3         Haveli       None       None       None      PK104  Azad Kashmir   
4          Kotli       None       None       None      PK105  Azad Kashmir   

  adm1_name1 adm1_name2 adm1_name3 adm1_pcode  ...    area_sqkm cod_versio  \
0       None       None       None        PK1  ...   694.416867       V_01   
1       None       None       None        PK1  ...  1212.947273       V_01   
2       None       None       None        PK1  ...   681.765079       V_01   
3       None       None       None        PK1  ...   550.893049       V_01   
4       None       None       None        PK1  ...  1599.117066       V_01   

  lang lang1 lang2 lang3     adm2_ref_n  center_lat center_lon

In [ ]:
joined = gpd.sjoin(
    gdf_loss,
    pk,
    how="left",
    predicate="within"
)

In [ ]:
print(joined.columns.tolist())
print(joined.head())

['system:index', 'gain', 'latitude', 'longitude', 'loss', 'lossyear', 'treecover2000', 'year', '.geo', 'geometry', 'index_right', 'adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'cod_versio', 'lang', 'lang1', 'lang2', 'lang3', 'adm2_ref_n', 'center_lat', 'center_lon']
   system:index  gain   latitude  longitude  loss  lossyear  treecover2000  \
0             0     0  36.382706  74.347507     1        21             76   
1             1     0  36.379741  74.345890     1        19             66   
2             2     0  36.373273  74.339961     1        19             74   
3             3     0  36.373004  74.340500     1        19             74   
4             4     0  36.266014  72.236017     1         8             45   

   year                                               .geo  \
0  2

In [ ]:
joined.drop(columns="geometry").to_csv(
    "/content/drive/MyDrive/forest_loss_location.csv",
    index=False
)